Copyright 2026, Battelle Energy Alliance, LLC, ALL RIGHTS RESERVED

In [1]:
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
os.chdir("../../..")
from HelpingFunctions import ERCOTProcessor
from HelpingFunctions import WeatherProcessing
from HelpingFunctions import FeatureEngineering
from HelpingFunctions import ForecastingHelpers

import onnxruntime as ort
ort.set_default_logger_severity(4)

In [2]:
import os 
os.getcwd()

'/home/ortild/Amaranth/opensourcegridmodeling'

Data Preprocessing for RT Data - Setting Up Model Inputs

In [3]:
full_df = pd.read_csv('ElectricityDemandAustinTX/LoadForecastingAttacks/full_data.csv', parse_dates=['time'], index_col=['time'])
hourly_res_norm = ForecastingHelpers.hourlyresiduals(full_df)

/home/ortild/Amaranth/opensourcegridmodeling/ElectricityDemandAustinTX/LoadForecastingAttacks/HelpingFunctions/ForecastingHelpers.py:74: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  hourly_res_norm['load'] = df_norm['load'].groupby(pd.Grouper(freq='M')).transform(lambda x: x - x.mean())


In [4]:
# train-validate-test split
train = hourly_res_norm[:'2014']
validate = hourly_res_norm['2015':'2016']
test = hourly_res_norm['2017':]

# setup training variables 
exog_tr = train.iloc[:,1:].values
ar_tr = train['load'].shift().bfill().values[:,None]
X_tr = np.hstack([ar_tr, exog_tr])
y_tr = train['load'].values

# setup validation variables
exog_val = validate.iloc[:,1:].values
y_val = validate['load'].values

# setup testing variables
exog_te = test.iloc[:,1:].values
ar_test = test['load'].shift().bfill().values[:,None]
y_test = test['load'].values
X_test = np.hstack([ar_test, exog_te])

# setup miscellaneous variables
yp_full = hourly_res_norm.loc[:'2016','load']
yp_val = hourly_res_norm.loc['2015':'2016','load']
yp_te = hourly_res_norm.loc['2017':,'load']
y_init_val = np.hstack([y_tr[-1], validate.iloc[167::168,0].values])
y_init_te = np.hstack([y_val[-1], test.iloc[167::168,0].values])

In [5]:
# Create input values
manualX = pd.DataFrame(X_test, columns=["time","temp","humd", "wnsp", "tstorm", "clear", "fog", "rain", "cloud", "snow","weekday", "sin_hr", "cos_hr"])
print(manualX.info())
print(manualX.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70128 entries, 0 to 70127
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   time     70128 non-null  float64
 1   temp     70128 non-null  float64
 2   humd     70128 non-null  float64
 3   wnsp     70128 non-null  float64
 4   tstorm   70128 non-null  float64
 5   clear    70128 non-null  float64
 6   fog      70128 non-null  float64
 7   rain     70128 non-null  float64
 8   cloud    70128 non-null  float64
 9   snow     70128 non-null  float64
 10  weekday  70128 non-null  float64
 11  sin_hr   70128 non-null  float64
 12  cos_hr   70128 non-null  float64
dtypes: float64(13)
memory usage: 7.0 MB
None
       time      temp      humd      wnsp  tstorm  clear  fog  rain  cloud  \
0 -0.099247  0.484848  0.930000  0.000000     0.0    0.0  1.0   0.0    0.0   
1 -0.099247  0.484848  0.930000  0.000000     0.0    0.0  1.0   0.0    0.0   
2 -0.095466  0.484848  0.930000  0.242424

Linear Regression Model

In [8]:
# getting and saving outputs for model inversion with inputs and outputs
import onnx
import onnxruntime as rt
onnx_model =  onnx.load("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/linear_no_mapie.onnx")
model_inv = onnx_model
session = rt.InferenceSession("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/linear_no_mapie.onnx")
input_name = session.get_inputs()[0].name
input_data = np.array(manualX, dtype=np.double) 
inputs = {input_name: input_data}
outputs = session.run(None, inputs)
print(outputs)

[array([[-0.117213  ],
       [-0.11572873],
       [-0.10967113],
       ...,
       [-0.0201042 ],
       [-0.03743246],
       [-0.0518272 ]], shape=(70128, 1))]


In [9]:
out_vals = pd.DataFrame(outputs[0])
out_vals.to_csv("ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/hourly_lr_results.csv")
# Saved results file has been resampled with monthly variables. Can either resample inputs to also be monthly, or run inputs through model to get outputs, resave, and use those.
results_df = pd.read_csv('ElectricityDemandAustinTX/LoadForecastingAttacks/SaveFiles/hourly_lr_results.csv')
data_df = pd.concat([manualX, results_df["0"]], axis = 1)
data_df.head(5)

,time,temp,humd,wnsp,tstorm,clear,fog,rain,cloud,snow,weekday,sin_hr,cos_hr,0
0,-0.099247,0.484848,0.930000,0.000000,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.000000,1.000000,-0.117213
1,-0.099247,0.484848,0.930000,0.000000,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.258819,0.965926,-0.115729
2,-0.095466,0.484848,0.930000,0.242424,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.500000,0.866025,-0.109671
3,-0.121121,0.486975,0.928421,0.229665,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.707107,0.707107,-0.129537
4,-0.130643,0.489102,0.926842,0.216906,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.866025,0.500000,-0.133286


In [10]:
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

y = data_df["0"].to_numpy()
x = data_df.drop(columns="0").to_numpy()
X_train, X_test, y_train, y_test = train_test_split(x, y, random_state=42, train_size = .7)

model = MLPRegressor(hidden_layer_sizes=(500, 3), max_iter=1000)
# Train the model
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
print(f"Mean Absolute Error (MAE): {mae}")

Mean Absolute Error (MAE): 0.0015629452927514418
